# Projeto final: previsão de churn

## Fase 1. Análise exploratória (parte 1)

Carregamento da base, tamanho, tipos de dados e sumário estatístico.

In [ ]:
from pathlib import Path

from imblearn.under_sampling import RandomUnderSampler
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

sns.set_theme(style="whitegrid")

DATA_PATH = Path("../datasets/ecommerce_dataset.csv")

df = pd.read_csv(DATA_PATH)
df.head()

### Tamanho da base

In [ ]:
n_linhas, n_colunas = df.shape
print(f"Linhas: {n_linhas}")
print(f"Colunas: {n_colunas}")

### Tipos de dados

In [ ]:
df.info()

In [ ]:
df.dtypes

### Sumário estatístico

In [ ]:
df.describe()

In [ ]:
df.describe(include=["object", "string"])

## Fase 1. Análise exploratória (parte 2)

Nesta parte, vamos medir o desbalanceamento do alvo, observar a distribuição do tempo de relacionamento e calcular as correlações de Pearson entre as variáveis numéricas.

### Distribuição da variável-alvo

In [ ]:
distribuicao_churn = pd.DataFrame(
    {
        "clientes": df["Churn"].value_counts().sort_index(),
        "percentual": df["Churn"].value_counts(normalize=True).sort_index().mul(100),
    }
)
distribuicao_churn.index = ["Permaneceu (0)", "Saiu (1)"]
distribuicao_churn.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.countplot(
    data=df,
    x="Churn",
    hue="Churn",
    palette={0: "#4C78A8", 1: "#E45756"},
    legend=False,
    ax=ax,
)
ax.bar_label(ax.containers[0])
ax.bar_label(ax.containers[1])
ax.set(
    title="Distribuição da variável-alvo",
    xlabel="Churn (0 = permaneceu, 1 = saiu)",
    ylabel="Clientes",
)
plt.show()

### Tempo de relacionamento por classe

In [ ]:
df.groupby("Churn")["Tenure"].agg(["count", "mean", "median"]).round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.histplot(
    data=df,
    x="Tenure",
    hue="Churn",
    bins=30,
    kde=True,
    multiple="layer",
    alpha=0.45,
    palette={0: "#4C78A8", 1: "#E45756"},
    ax=ax,
)
ax.set(
    title="Distribuição do tempo de relacionamento por classe",
    xlabel="Tempo de relacionamento",
    ylabel="Clientes",
)
plt.show()

### Correlação entre variáveis numéricas

`CustomerID` fica fora do cálculo porque é apenas um identificador.

In [ ]:
variaveis_numericas = df.select_dtypes(include="number").drop(columns="CustomerID")
correlacao = variaveis_numericas.corr(method="pearson")

fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(
    correlacao,
    cmap="coolwarm",
    center=0,
    annot=True,
    fmt=".2f",
    linewidths=0.5,
    ax=ax,
)
ax.set_title("Correlação de Pearson entre as variáveis numéricas")
plt.tight_layout()
plt.show()

In [ ]:
(
    correlacao["Churn"]
    .drop("Churn")
    .sort_values(key=lambda valores: valores.abs(), ascending=False)
    .head(5)
)

### Leitura inicial

A base está desbalanceada: só 16,84% dos clientes saíram. A acurácia sozinha pode passar uma impressão errada.

`Tenure` é o sinal mais forte até agora, com correlação de -0,349. Quem saiu tem mediana 1; quem ficou, 10. Reclamação também puxa o churn, com correlação de 0,250, mas nenhum desses fatores sozinho fecha o caso.

Sete colunas numéricas têm valores ausentes, inclusive `Tenure`. Isso precisa ser resolvido antes da engenharia de atributos e da modelagem. O split vai manter a proporção das classes, e o balanceamento ficará só no treino.

# Fase 2. Limpeza e tratamento

A limpeza começa pelas duplicatas e pelos valores ausentes. Depois, usamos boxplots e o intervalo interquartil para entender os extremos antes de decidir se vale alterar algum registro.

## Registros duplicados

In [ ]:
quantidade_duplicadas = int(df.duplicated().sum())
print(f"Linhas duplicadas: {quantidade_duplicadas}")

df_limpo = df.drop_duplicates().copy()
print(f"Formato após a remoção: {df_limpo.shape}")

A busca não encontrou duplicatas. Ainda assim, mantemos `drop_duplicates()` no fluxo para que a limpeza continue válida se a fonte mudar.

## Valores ausentes

Vamos comparar quantidade de nulos, assimetria, média e mediana. Quanto mais distante de zero estiver a assimetria, menos a média representa o centro da distribuição.

In [ ]:
nulos_por_coluna = df_limpo.isna().sum()
colunas_com_nulos = nulos_por_coluna[nulos_por_coluna > 0].index.tolist()

resumo_nulos = pd.DataFrame(
    {
        "ausentes": nulos_por_coluna[colunas_com_nulos],
        "percentual": nulos_por_coluna[colunas_com_nulos].div(len(df_limpo)).mul(100),
        "assimetria": df_limpo[colunas_com_nulos].skew(),
        "media": df_limpo[colunas_com_nulos].mean(),
        "mediana": df_limpo[colunas_com_nulos].median(),
    }
).sort_values("ausentes", ascending=False)

resumo_nulos.round(2)

`HourSpendOnApp` é praticamente simétrica, com assimetria de -0,03, então vamos preencher seus nulos com a média. As outras seis colunas têm cauda à direita e valores extremos. Nelas, a mediana é uma escolha mais segura e muda menos com esses extremos.

In [ ]:
estrategias_imputacao = {
    "Tenure": "mediana",
    "WarehouseToHome": "mediana",
    "HourSpendOnApp": "media",
    "OrderAmountHikeFromlastYear": "mediana",
    "CouponUsed": "mediana",
    "OrderCount": "mediana",
    "DaySinceLastOrder": "mediana",
}

registro_imputacao = []

for coluna, estrategia in estrategias_imputacao.items():
    if estrategia == "media":
        valor = df_limpo[coluna].mean()
    else:
        valor = df_limpo[coluna].median()

    df_limpo[coluna] = df_limpo[coluna].fillna(valor)
    registro_imputacao.append(
        {"coluna": coluna, "estrategia": estrategia, "valor": valor}
    )

pd.DataFrame(registro_imputacao).round({"valor": 2})

In [ ]:
print(f"Valores ausentes após a imputação: {int(df_limpo.isna().sum().sum())}")

## Outliers

O boxplot ajuda a localizar valores distantes do miolo da distribuição. Para contar esses pontos, usamos a regra de 1,5 vezes o intervalo interquartil.

In [ ]:
colunas_para_boxplot = [
    "Tenure",
    "WarehouseToHome",
    "HourSpendOnApp",
    "OrderAmountHikeFromlastYear",
    "CouponUsed",
    "OrderCount",
    "DaySinceLastOrder",
    "CashbackAmount",
]

fig, eixos = plt.subplots(2, 4, figsize=(18, 8))

for coluna, eixo in zip(colunas_para_boxplot, eixos.flat):
    sns.boxplot(x=df_limpo[coluna], color="#4C78A8", ax=eixo)
    eixo.set_title(coluna)
    eixo.set_xlabel("")

fig.suptitle("Distribuição das variáveis numéricas", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
registro_outliers = []

for coluna in colunas_para_boxplot:
    primeiro_quartil = df_limpo[coluna].quantile(0.25)
    terceiro_quartil = df_limpo[coluna].quantile(0.75)
    intervalo_interquartil = terceiro_quartil - primeiro_quartil
    limite_inferior = primeiro_quartil - 1.5 * intervalo_interquartil
    limite_superior = terceiro_quartil + 1.5 * intervalo_interquartil
    fora_dos_limites = (
        (df_limpo[coluna] < limite_inferior)
        | (df_limpo[coluna] > limite_superior)
    )
    quantidade = int(fora_dos_limites.sum())

    registro_outliers.append(
        {
            "coluna": coluna,
            "limite_inferior": limite_inferior,
            "limite_superior": limite_superior,
            "outliers": quantidade,
            "percentual": quantidade / len(df_limpo) * 100,
        }
    )

resumo_outliers = pd.DataFrame(registro_outliers).sort_values(
    "percentual", ascending=False
)
resumo_outliers.round(2)

### Decisão sobre os outliers

Vamos manter os valores. A regra do IQR marcou 12,49% de `OrderCount` e 11,17% de `CouponUsed`, mas são contagens plausíveis de clientes mais ativos. Cortá-las apagaria justamente parte do comportamento que queremos estudar. Os extremos das outras colunas também não bastam, sozinhos, para afirmar que houve erro de cadastro.

Essa escolha exige cuidado com o KNN, porque distâncias grandes pesam no resultado. Na etapa de modelagem, o escalonamento será ajustado apenas no treino e vamos comparar as métricas de treino e teste. A Árvore de Decisão é menos sensível a esses extremos.

## Resultado da limpeza

Não havia duplicatas para remover. Os nulos foram preenchidos com média em `HourSpendOnApp` e mediana nas outras seis colunas. Os outliers foram mantidos porque parecem representar comportamento real, não erro confirmado.

In [ ]:
pd.Series(
    {
        "linhas": len(df_limpo),
        "colunas": df_limpo.shape[1],
        "duplicatas": int(df_limpo.duplicated().sum()),
        "valores_ausentes": int(df_limpo.isna().sum().sum()),
    },
    name="resultado",
)

# Fase 3. Engenharia de atributos

`CashbackAmount` mostra o cashback total, mas não diz quanto cada compra devolveu ao cliente. Para enxergar essa relação, vamos criar `cashback_por_pedido`.

## Validação das colunas de origem

Os nulos já foram tratados na fase anterior. Falta confirmar que `OrderCount` não contém zero, pois isso tornaria a divisão inválida.

In [ ]:
validacao_origens = pd.Series(
    {
        "nulos_em_cashback": int(df_limpo["CashbackAmount"].isna().sum()),
        "nulos_em_pedidos": int(df_limpo["OrderCount"].isna().sum()),
        "pedidos_iguais_a_zero": int(df_limpo["OrderCount"].eq(0).sum()),
    },
    name="quantidade",
)
validacao_origens

In [ ]:
if validacao_origens["pedidos_iguais_a_zero"] > 0:
    raise ValueError("OrderCount contém zero; não é possível calcular cashback por pedido.")

df_limpo["cashback_por_pedido"] = df_limpo["CashbackAmount"].div(
    df_limpo["OrderCount"]
)

df_limpo[
    ["CashbackAmount", "OrderCount", "cashback_por_pedido"]
].head()

## Conferência da nova coluna

In [ ]:
valores_calculados = df_limpo["cashback_por_pedido"]

pd.Series(
    {
        "nulos": int(valores_calculados.isna().sum()),
        "infinitos": int(
            valores_calculados.isin([float("inf"), float("-inf")]).sum()
        ),
        "minimo": valores_calculados.min(),
        "maximo": valores_calculados.max(),
    },
    name="resultado",
).round(2)

In [ ]:
df_limpo["cashback_por_pedido"].describe().round(2)

In [ ]:
df_limpo.groupby("Churn")["cashback_por_pedido"].agg(
    ["count", "mean", "median"]
).round(2)

## Resultado

A nova coluna ficou sem nulos ou infinitos. A mediana é 88,50 para quem permaneceu e 77,50 para quem saiu. A diferença existe, mas é pequena demais para explicar o Churn por si só. Com isso, `cashback_por_pedido` entra como mais um sinal, junto com as outras colunas.

# Fase 4. Preparação dos dados (parte 1)

Nesta parte, vamos separar o alvo, reservar a base de teste e codificar as categorias. O balanceamento e o escalonamento ficam para a próxima parte.

## Ajuste dos nomes das categorias

A base usa nomes diferentes para a mesma coisa, como `Phone` e `Mobile Phone`. Se passarmos isso direto para o encoder, o modelo vai tratar os dois textos como categorias distintas. Vamos juntar esses aliases antes do split. O mapeamento é uma regra fixa e não usa a variável-alvo.

In [ ]:
df_modelagem = df_limpo.copy()

mapeamento_categorias = {
    "PreferredLoginDevice": {"Phone": "Mobile Phone"},
    "PreferredPaymentMode": {
        "CC": "Credit Card",
        "COD": "Cash on Delivery",
    },
    "PreferedOrderCat": {"Mobile": "Mobile Phone"},
}

for coluna, mapeamento in mapeamento_categorias.items():
    df_modelagem[coluna] = df_modelagem[coluna].replace(mapeamento)

pd.Series(
    {
        coluna: sorted(df_modelagem[coluna].unique().tolist())
        for coluna in mapeamento_categorias
    },
    name="categorias",
).to_frame()

## Preditores e alvo

`Churn` é o alvo. `CustomerID` fica fora dos preditores porque só identifica a linha e não descreve o comportamento do cliente.

In [ ]:
X = df_modelagem.drop(columns=["CustomerID", "Churn"])
y = df_modelagem["Churn"].copy()

colunas_categoricas = X.select_dtypes(include=["object", "string"]).columns.tolist()
colunas_numericas = [
    coluna for coluna in X.columns if coluna not in colunas_categoricas
]

pd.Series(
    {
        "preditores_originais": X.shape[1],
        "categoricas": len(colunas_categoricas),
        "numericas": len(colunas_numericas),
    },
    name="quantidade",
)

## Split estratificado

Reservamos 20% da base para teste. `stratify=y` mantém a proporção de churn nos dois conjuntos, e `random_state=42` permite repetir a mesma divisão.

In [ ]:
X_treino_bruto, X_teste_bruto, y_treino, y_teste = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print(f"Treino: {X_treino_bruto.shape}")
print(f"Teste: {X_teste_bruto.shape}")

In [ ]:
distribuicao_split = pd.DataFrame(
    {
        "base_completa": y.value_counts(normalize=True).sort_index(),
        "treino": y_treino.value_counts(normalize=True).sort_index(),
        "teste": y_teste.value_counts(normalize=True).sort_index(),
    }
).mul(100)

distribuicao_split.index = ["Permaneceu (0)", "Saiu (1)"]
distribuicao_split.round(2)

## Codificação das categorias

As cinco variáveis categóricas não têm ordem natural, então usamos one-hot encoding. O encoder aprende as categorias somente no treino. No teste, ele apenas aplica o mesmo mapeamento e ignora uma categoria nova caso ela apareça.

In [ ]:
codificador = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

categorias_treino = codificador.fit_transform(
    X_treino_bruto[colunas_categoricas]
)
categorias_teste = codificador.transform(
    X_teste_bruto[colunas_categoricas]
)

colunas_codificadas = codificador.get_feature_names_out(colunas_categoricas)

X_treino_categorico = pd.DataFrame(
    categorias_treino,
    columns=colunas_codificadas,
    index=X_treino_bruto.index,
)
X_teste_categorico = pd.DataFrame(
    categorias_teste,
    columns=colunas_codificadas,
    index=X_teste_bruto.index,
)

In [ ]:
X_treino = pd.concat(
    [X_treino_bruto[colunas_numericas], X_treino_categorico],
    axis=1,
)
X_teste = pd.concat(
    [X_teste_bruto[colunas_numericas], X_teste_categorico],
    axis=1,
)

print(f"Treino após o encoding: {X_treino.shape}")
print(f"Teste após o encoding: {X_teste.shape}")
X_treino.head()

## Conferência da preparação

In [ ]:
pd.DataFrame(
    {
        "linhas": [len(X_treino), len(X_teste)],
        "colunas": [X_treino.shape[1], X_teste.shape[1]],
        "nulos": [
            int(X_treino.isna().sum().sum()),
            int(X_teste.isna().sum().sum()),
        ],
        "churn_percentual": [
            y_treino.mean() * 100,
            y_teste.mean() * 100,
        ],
    },
    index=["treino", "teste"],
).round(2)

## Resultado

O treino ficou com 4.504 clientes e o teste com 1.126. A taxa de churn permaneceu perto de 16,84% nos dois conjuntos. Depois do one-hot encoding, cada base tem 31 preditores numéricos e nenhum valor ausente.

A base de teste não participou do ajuste do encoder. Ela continua separada para medir, mais adiante, como os modelos se comportam com dados que não viram.

# Fase 4. Preparação dos dados (parte 2)

Agora vamos balancear somente o treino e montar dois fluxos. O KNN recebe as variáveis contínuas escalonadas; a Árvore de Decisão usa os valores originais.

## Balanceamento do treino

Vamos usar `RandomUnderSampler`. Ele reduz a classe majoritária sem inventar combinações nas colunas one-hot. O custo é abrir mão de parte dos clientes que permaneceram, então esse efeito precisa aparecer na comparação entre treino e teste.

A base de teste fica como está. Balanceá-la criaria uma avaliação diferente do cenário real.

In [ ]:
balanceador = RandomUnderSampler(random_state=42)
X_treino_balanceado, y_treino_balanceado = balanceador.fit_resample(
    X_treino,
    y_treino,
)

pd.DataFrame(
    {
        "treino_antes": y_treino.value_counts().sort_index(),
        "treino_depois": y_treino_balanceado.value_counts().sort_index(),
        "teste_inalterado": y_teste.value_counts().sort_index(),
    },
    index=[0, 1],
)

O treino passou de 4.504 para 1.516 registros, com 758 clientes em cada classe. O teste continua com 1.126 registros e a proporção original de churn.

## Fluxo da Árvore de Decisão

A árvore compara cortes, não distâncias. Mudar a escala não altera a ordem dos valores, então ela recebe o treino balanceado sem `StandardScaler`.

In [ ]:
X_treino_arvore = X_treino_balanceado.copy()
X_teste_arvore = X_teste.copy()
y_treino_arvore = y_treino_balanceado.copy()
y_teste_arvore = y_teste.copy()

## Fluxo do KNN

No KNN, cada previsão depende de distância. Sem escalonamento, uma coluna com valores altos pode dominar as demais. O `StandardScaler` é ajustado nas variáveis quantitativas do treino balanceado e apenas aplicado ao teste. `CityTier`, `SatisfactionScore`, `Complain` e as colunas one-hot continuam na escala original.

In [ ]:
colunas_para_escalar = [
    "Tenure",
    "WarehouseToHome",
    "HourSpendOnApp",
    "NumberOfDeviceRegistered",
    "NumberOfAddress",
    "OrderAmountHikeFromlastYear",
    "CouponUsed",
    "OrderCount",
    "DaySinceLastOrder",
    "CashbackAmount",
    "cashback_por_pedido",
]

escalonador = StandardScaler()

X_treino_knn = X_treino_balanceado.copy()
X_teste_knn = X_teste.copy()

X_treino_knn[colunas_para_escalar] = escalonador.fit_transform(
    X_treino_balanceado[colunas_para_escalar]
)
X_teste_knn[colunas_para_escalar] = escalonador.transform(
    X_teste[colunas_para_escalar]
)

y_treino_knn = y_treino_balanceado.copy()
y_teste_knn = y_teste.copy()

## Conferência dos fluxos

In [ ]:
resumo_escalonamento = pd.DataFrame(
    {
        "media_no_treino": X_treino_knn[colunas_para_escalar].mean(),
        "desvio_no_treino": X_treino_knn[colunas_para_escalar].std(ddof=0),
    }
)
resumo_escalonamento.round(2)

In [ ]:
colunas_one_hot = colunas_codificadas.tolist()
colunas_nao_escaladas = [
    "CityTier",
    "SatisfactionScore",
    "Complain",
    *colunas_one_hot,
]

pd.Series(
    {
        "categorias_permaneceram_iguais": X_treino_knn[
            colunas_nao_escaladas
        ].equals(X_treino_balanceado[colunas_nao_escaladas]),
        "arvore_permaneceu_sem_escala": X_treino_arvore.equals(
            X_treino_balanceado
        ),
        "teste_permaneceu_desbalanceado": round(y_teste.mean() * 100, 2),
    },
    name="validacao",
)

In [ ]:
pd.DataFrame(
    {
        "treino_linhas": [len(X_treino_knn), len(X_treino_arvore)],
        "teste_linhas": [len(X_teste_knn), len(X_teste_arvore)],
        "preditores": [X_treino_knn.shape[1], X_treino_arvore.shape[1]],
        "nulos_treino": [
            int(X_treino_knn.isna().sum().sum()),
            int(X_treino_arvore.isna().sum().sum()),
        ],
    },
    index=["KNN", "Arvore"],
)

## Resultado

Os dois modelos usarão o mesmo treino balanceado e o mesmo teste original. A diferença está na escala: o KNN recebe 11 variáveis quantitativas padronizadas, enquanto a árvore trabalha com os valores sem transformação.

O teste não entrou no balanceamento nem no ajuste do scaler. Ele segue representando a distribuição real que os modelos devem enfrentar.

# Fase 5. Modelagem e overfitting

Vamos treinar KNN e Árvore de Decisão com vários hiperparâmetros. Em cada cenário, comparamos treino e teste. Se a métrica do treino sobe muito e a do teste não acompanha, o modelo está decorando.

A métrica principal para a escolha será o F1 da classe de churn no teste. Como o teste permanece desbalanceado, a acurácia sozinha engana. Ainda assim, o gap entre treino e teste entra na decisão, para evitar configurações que só memorizam.

In [ ]:
def avaliar_modelo(modelo, X_treino_modelo, y_treino_modelo, X_teste_modelo, y_teste_modelo):
    modelo.fit(X_treino_modelo, y_treino_modelo)
    previsao_treino = modelo.predict(X_treino_modelo)
    previsao_teste = modelo.predict(X_teste_modelo)

    return {
        "acuracia_treino": accuracy_score(y_treino_modelo, previsao_treino),
        "acuracia_teste": accuracy_score(y_teste_modelo, previsao_teste),
        "f1_treino": f1_score(y_treino_modelo, previsao_treino),
        "f1_teste": f1_score(y_teste_modelo, previsao_teste),
        "gap_f1": f1_score(y_treino_modelo, previsao_treino)
        - f1_score(y_teste_modelo, previsao_teste),
    }

## KNN

Testamos cinco valores de `n_neighbors`: 3, 5, 7, 9 e 11. K pequeno costuma se aproximar demais do treino; K maior suaviza a decisão.

In [ ]:
valores_k = [3, 5, 7, 9, 11]
resultados_knn = []

for k in valores_k:
    metricas = avaliar_modelo(
        KNeighborsClassifier(n_neighbors=k),
        X_treino_knn,
        y_treino_knn,
        X_teste_knn,
        y_teste_knn,
    )
    resultados_knn.append({"n_neighbors": k, **metricas})

tabela_knn = pd.DataFrame(resultados_knn)
tabela_knn.round(3)

In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(12, 4))

eixos[0].plot(
    tabela_knn["n_neighbors"],
    tabela_knn["f1_treino"],
    marker="o",
    label="Treino",
)
eixos[0].plot(
    tabela_knn["n_neighbors"],
    tabela_knn["f1_teste"],
    marker="o",
    label="Teste",
)
eixos[0].set(
    title="F1 do KNN por valor de K",
    xlabel="n_neighbors",
    ylabel="F1",
)
eixos[0].legend()

eixos[1].plot(
    tabela_knn["n_neighbors"],
    tabela_knn["gap_f1"],
    marker="o",
    color="#E45756",
)
eixos[1].set(
    title="Gap de F1 entre treino e teste",
    xlabel="n_neighbors",
    ylabel="Gap de F1",
)

plt.tight_layout()
plt.show()

Com K=3, o F1 de teste fica em 0,560, mas o gap sobe para 0,380. O modelo pega bem o treino e perde mais no teste. Em K=5, o F1 de teste quase não muda (0,558) e o gap cai para 0,353. A partir daí, o gap continua diminuindo, mas o F1 de teste também.

Escolhemos K=5. Ele mantém um desempenho próximo do melhor no teste e deixa o overfitting um pouco menos agressivo do que K=3.

## Árvore de Decisão

Testamos cinco profundidades: 3, 5, 7, 10 e sem limite (`None`). Árvore rasa perde detalhes. Árvore profunda ou ilimitada memoriza o treino com facilidade.

In [ ]:
valores_profundidade = [3, 5, 7, 10, None]
resultados_arvore = []

for profundidade in valores_profundidade:
    metricas = avaliar_modelo(
        DecisionTreeClassifier(max_depth=profundidade, random_state=42),
        X_treino_arvore,
        y_treino_arvore,
        X_teste_arvore,
        y_teste_arvore,
    )
    resultados_arvore.append({"max_depth": profundidade, **metricas})

tabela_arvore = pd.DataFrame(resultados_arvore)
tabela_arvore["max_depth_rotulo"] = tabela_arvore["max_depth"].fillna("None")
tabela_arvore.round(3)

In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(12, 4))
rotulos = tabela_arvore["max_depth_rotulo"].astype(str)

eixos[0].plot(rotulos, tabela_arvore["f1_treino"], marker="o", label="Treino")
eixos[0].plot(rotulos, tabela_arvore["f1_teste"], marker="o", label="Teste")
eixos[0].set(
    title="F1 da Árvore por profundidade",
    xlabel="max_depth",
    ylabel="F1",
)
eixos[0].legend()

eixos[1].plot(rotulos, tabela_arvore["gap_f1"], marker="o", color="#E45756")
eixos[1].set(
    title="Gap de F1 entre treino e teste",
    xlabel="max_depth",
    ylabel="Gap de F1",
)

plt.tight_layout()
plt.show()

Sem limite de profundidade, o treino chega a F1 1,000 e o gap sobe para 0,345. A árvore decorou. Em `max_depth=10`, o F1 de teste sobe para 0,665, mas o gap ainda fica alto (0,299). Em `max_depth=7`, o F1 de teste fica em 0,635 e o gap cai para 0,269, o menor entre as configurações competitivas.

Escolhemos `max_depth=7`. Ela generaliza melhor do que a árvore ilimitada e evita o salto de complexidade de profundidade 10.

## Configurações selecionadas

Refazemos o treino das melhores configurações e guardamos as previsões para a fase de avaliação.

In [ ]:
melhor_knn = KNeighborsClassifier(n_neighbors=5)
melhor_knn.fit(X_treino_knn, y_treino_knn)
previsao_treino_knn = melhor_knn.predict(X_treino_knn)
previsao_teste_knn = melhor_knn.predict(X_teste_knn)

melhor_arvore = DecisionTreeClassifier(max_depth=7, random_state=42)
melhor_arvore.fit(X_treino_arvore, y_treino_arvore)
previsao_treino_arvore = melhor_arvore.predict(X_treino_arvore)
previsao_teste_arvore = melhor_arvore.predict(X_teste_arvore)

pd.DataFrame(
    {
        "hiperparametro": ["n_neighbors=5", "max_depth=7"],
        "f1_treino": [
            f1_score(y_treino_knn, previsao_treino_knn),
            f1_score(y_treino_arvore, previsao_treino_arvore),
        ],
        "f1_teste": [
            f1_score(y_teste_knn, previsao_teste_knn),
            f1_score(y_teste_arvore, previsao_teste_arvore),
        ],
        "acuracia_teste": [
            accuracy_score(y_teste_knn, previsao_teste_knn),
            accuracy_score(y_teste_arvore, previsao_teste_arvore),
        ],
        "gap_f1": [
            f1_score(y_treino_knn, previsao_treino_knn)
            - f1_score(y_teste_knn, previsao_teste_knn),
            f1_score(y_treino_arvore, previsao_treino_arvore)
            - f1_score(y_teste_arvore, previsao_teste_arvore),
        ],
    },
    index=["KNN", "Arvore"],
).round(3)

## Resultado

No KNN, K=5 equilibrou o F1 de teste e o gap de overfitting. Na Árvore, `max_depth=7` foi o ponto em que o teste ainda melhorava sem o salto de memorização visto em profundidade 10 e na árvore ilimitada.

Nesta fase, a Árvore ficou à frente no F1 de teste: 0,635 contra 0,558 do KNN, e com gap menor (0,269 contra 0,353). No recall da classe Churn, porém, o KNN é maior: 0,895 contra 0,858 da Árvore. O F1 da Árvore vem de uma precisão mais alta, não de pegar mais churners. Nenhum modelo domina em tudo.

# Fase 6. Avaliação e veredito de negócio

Testo aqui as configurações escolhidas no conjunto de teste original, com foco na classe de churn: os clientes que queremos alcançar antes que saiam.

## Relatórios de classificação

Foco na classe `Churn`: recall responde quantos clientes em risco foram encontrados; precisão mostra quantos alertas de churn estavam certos.

In [ ]:
nomes_classes = ["Permaneceu", "Churn"]

print("KNN (n_neighbors=5)\n")
print(
    classification_report(
        y_teste_knn,
        previsao_teste_knn,
        target_names=nomes_classes,
        digits=3,
    )
)

print("Árvore de Decisão (max_depth=7)\n")
print(
    classification_report(
        y_teste_arvore,
        previsao_teste_arvore,
        target_names=nomes_classes,
        digits=3,
    )
)

In [ ]:
def resumir_avaliacao(y_real, y_previsto):
    verdadeiro_negativo, falso_positivo, falso_negativo, verdadeiro_positivo = (
        confusion_matrix(y_real, y_previsto, labels=[0, 1]).ravel()
    )

    return {
        "acuracia": accuracy_score(y_real, y_previsto),
        "precisao_churn": precision_score(y_real, y_previsto),
        "recall_churn": recall_score(y_real, y_previsto),
        "f1_churn": f1_score(y_real, y_previsto),
        "falso_positivo": falso_positivo,
        "falso_negativo": falso_negativo,
        "verdadeiro_positivo": verdadeiro_positivo,
        "verdadeiro_negativo": verdadeiro_negativo,
    }

comparacao_final = pd.DataFrame(
    {
        "KNN": resumir_avaliacao(y_teste_knn, previsao_teste_knn),
        "Arvore": resumir_avaliacao(y_teste_arvore, previsao_teste_arvore),
    }
).T

comparacao_final.round(3)

## Matrizes de confusão

Falso positivo é um cupom enviado sem necessidade. Falso negativo é um cliente em risco que não recebe a tentativa de retenção.

In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(12, 5))

ConfusionMatrixDisplay.from_predictions(
    y_teste_knn,
    previsao_teste_knn,
    labels=[0, 1],
    display_labels=nomes_classes,
    cmap="Blues",
    colorbar=False,
    ax=eixos[0],
)
eixos[0].set_title("KNN (n_neighbors=5)")

ConfusionMatrixDisplay.from_predictions(
    y_teste_arvore,
    previsao_teste_arvore,
    labels=[0, 1],
    display_labels=nomes_classes,
    cmap="Greens",
    colorbar=False,
    ax=eixos[1],
)
eixos[1].set_title("Árvore (max_depth=7)")

plt.tight_layout()
plt.show()

O KNN encontra 170 dos 190 clientes que saíram e deixa 20 passar. Para isso, também sinaliza 249 clientes que permaneceriam. A Árvore encontra 163 churners, deixa 27 passar e reduz os falsos positivos para 160.

O falso negativo é o erro mais caro por caso: a plataforma não tenta reter alguém que realmente vai sair. Mas a escolha entre os modelos depende do tamanho dessa perda em relação ao custo do cupom.

In [ ]:
falsos_positivos_evitados = int(
    comparacao_final.loc["KNN", "falso_positivo"]
    - comparacao_final.loc["Arvore", "falso_positivo"]
)
falsos_negativos_adicionais = int(
    comparacao_final.loc["Arvore", "falso_negativo"]
    - comparacao_final.loc["KNN", "falso_negativo"]
)
razao_custo_equilibrio = (
    falsos_positivos_evitados / falsos_negativos_adicionais
)

pd.Series(
    {
        "cupons_desnecessarios_evitados_pela_arvore": falsos_positivos_evitados,
        "churners_adicionais_perdidos_pela_arvore": falsos_negativos_adicionais,
        "razao_de_custo_para_preferir_knn": razao_custo_equilibrio,
    },
    name="impacto",
).round(2)

## Veredito

Eu colocaria a Árvore de Decisão com `max_depth=7` em um piloto controlado. Ela tem F1 maior (0,635 contra 0,558), precisão melhor (0,505 contra 0,406) e reduz 89 cupons desnecessários. Em troca, perde 7 churners a mais do que o KNN.

O KNN só passa a compensar se perder um churner custar mais de 12,71 vezes o valor de um cupom. Como o projeto não traz esses custos, a Árvore é a escolha mais equilibrada com os dados disponíveis.

Antes de produção, faltam o custo médio do cupom e o valor esperado de um cliente retido. Também precisamos testar o limiar de decisão, acompanhar mudanças no comportamento e repetir a avaliação com novas levas de clientes.